# Introduction to Data Science 2026

# Week 4

In this week's exercise, we look at prompting and zero- and few-shot task settings. Below is a text generation example from https://github.com/TurkuNLP/intro-to-nlp/blob/master/text_generation_pipeline_example.ipynb demonstrating how to load a text generation pipeline with a pre-trained model and generate text with a given prompt. Your task is to load a similar pre-trained generative model and assess whether the model succeeds at a set of tasks in zero-shot, one-shot, and two-shot settings.

**Note: Downloading and running the pre-trained model locally may take some time. Alternatively, you can open and run this notebook on [Google Colab](https://colab.research.google.com/), as assumed in the following example.**

## Text generation example

This is a brief example of how to run text generation with a causal language model and `pipeline`.

Install [transformers](https://huggingface.co/docs/transformers/index) python package. This will be used to load the model and tokenizer and to run generation.

In [3]:
!pip install --quiet transformers

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

Load a generative model and its tokenizer. You can substitute any other generative model name here (e.g. [other TurkuNLP GPT-3 models](https://huggingface.co/models?sort=downloads&search=turkunlp%2Fgpt3)), but note that Colab may have issues running larger models. 

In [5]:
MODEL_NAME = 'TurkuNLP/gpt3-finnish-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/562 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/218 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.23M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 3.53GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.53GB            

model.safetensors: downloading bytes:           |  0.00B            

Instantiate a text generation pipeline using the tokenizer and model.

In [6]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [7]:
output = pipe('Terve, miten menee?', max_new_tokens=25)

print(output)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Terve, miten menee?”\n”Hyvin menee.”\n”Entä sinä?”\n”Hyvin menee.”\n”Siinä tapauksessa...”\n”'}]


Just print the text

In [8]:
print(output[0]['generated_text'])

Terve, miten menee?”
”Hyvin menee.”
”Entä sinä?”
”Hyvin menee.”
”Siinä tapauksessa...”
”


We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:

In [9]:
output = pipe(
    'Terve, miten menee?',
    do_sample=True,
    temperature=10.0,
    max_new_tokens=25
)

print(output[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Terve, miten menee? Ei meillä mitään sen mullistapaa tänään ohjelmassa ollutkaan vaikka oltiin täällä meillä yötä miehen loman alettua kun lapset halusi mun vanhempien seuraan mummolaan koko


## Exercise 1

Your task is to assess whether a generative model succeeds in the following tasks in zero-shot, one-shot, and two-shot settings:

- binary sentiment classification (positive / negative)

- person name recognition

- two-digit addition (e.g. 11 + 22 = 33)

For example, for assessing whether a generative model can name capital cities, we could use the following prompts:

- zero-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- one-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- two-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Denmark?\
	>Answer: Copenhagen
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""

You can do the tasks either in English or Finnish and use a generative model of your choice from the Hugging Face models repository, for example the following models:

- English: `gpt2-large`
- Finnish: `TurkuNLP/gpt3-finnish-large`

You can either come up with your own instructions for the tasks or use the following:

- English:
	- binary sentiment classification: "Do the following texts express a positive or negative sentiment?"
	- person name recognition: "List the person names occurring in the following texts."
	- two-digit addition: "This is a first grade math exam."
- Finnish:
	- binary sentiment classification: "Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?"
	- person name recognition: "Listaa seuraavissa teksteissä mainitut henkilönnimet."
	- two-digit addition: "Tämä on ensimmäisen luokan matematiikan koe."

Come up with at least two test cases for each of the three tasks, and come up with your own one- and two-shot examples.

In [10]:
MODEL_NAME = 'gpt2-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.25GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [11]:
def generate(prompt, max_new_tokens=15):
    output = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return output[0]['generated_text']

In [12]:
sentiment_instruction = "Do the following texts express a positive or negative sentiment?"

# ---- Test case A ----
text_a = "I can't believe how amazing this movie was, I loved every second of it."

prompt_a_zero = f"""{sentiment_instruction}

Text: {text_a}
Sentiment:"""

prompt_a_one = f"""{sentiment_instruction}

Text: The food was cold and the service was terrible.
Sentiment: Negative

Text: {text_a}
Sentiment:"""

prompt_a_two = f"""{sentiment_instruction}

Text: The food was cold and the service was terrible.
Sentiment: Negative

Text: What a wonderful, sunny day at the park with my friends.
Sentiment: Positive

Text: {text_a}
Sentiment:"""

print("=== Test case A (expected: Positive) ===")
print("ZERO-SHOT:", generate(prompt_a_zero))
print("\nONE-SHOT:", generate(prompt_a_one))
print("\nTWO-SHOT:", generate(prompt_a_two))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Test case A (expected: Positive) ===


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ZERO-SHOT: Do the following texts express a positive or negative sentiment?

Text: I can't believe how amazing this movie was, I loved every second of it.
Sentiment: I can't believe how amazing this movie was, I loved every second of


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ONE-SHOT: Do the following texts express a positive or negative sentiment?

Text: The food was cold and the service was terrible.
Sentiment: Negative

Text: I can't believe how amazing this movie was, I loved every second of it.
Sentiment: Positive

Text: I'm so glad I went to see this movie

TWO-SHOT: Do the following texts express a positive or negative sentiment?

Text: The food was cold and the service was terrible.
Sentiment: Negative

Text: What a wonderful, sunny day at the park with my friends.
Sentiment: Positive

Text: I can't believe how amazing this movie was, I loved every second of it.
Sentiment: Positive

Text: I love the way you look in that dress.


In [13]:
# ---- Test case B ----
text_b = "This was the worst customer service experience I have ever had in my life."

prompt_b_zero = f"""{sentiment_instruction}

Text: {text_b}
Sentiment:"""

prompt_b_one = f"""{sentiment_instruction}

Text: What a wonderful, sunny day at the park with my friends.
Sentiment: Positive

Text: {text_b}
Sentiment:"""

prompt_b_two = f"""{sentiment_instruction}

Text: The food was cold and the service was terrible.
Sentiment: Negative

Text: What a wonderful, sunny day at the park with my friends.
Sentiment: Positive

Text: {text_b}
Sentiment:"""

print("=== Test case B (expected: Negative) ===")
print("ZERO-SHOT:", generate(prompt_b_zero))
print("\nONE-SHOT:", generate(prompt_b_one))
print("\nTWO-SHOT:", generate(prompt_b_two))

[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Test case B (expected: Negative) ===


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ZERO-SHOT: Do the following texts express a positive or negative sentiment?

Text: This was the worst customer service experience I have ever had in my life.
Sentiment: I'm not sure what to say.

Text: I'm not


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ONE-SHOT: Do the following texts express a positive or negative sentiment?

Text: What a wonderful, sunny day at the park with my friends.
Sentiment: Positive

Text: This was the worst customer service experience I have ever had in my life.
Sentiment: Negative

Text: I am so glad I did not have to go

TWO-SHOT: Do the following texts express a positive or negative sentiment?

Text: The food was cold and the service was terrible.
Sentiment: Negative

Text: What a wonderful, sunny day at the park with my friends.
Sentiment: Positive

Text: This was the worst customer service experience I have ever had in my life.
Sentiment: Positive

Text: I was so happy to see my friend.



In [14]:
name_instruction = "List the person names occurring in the following texts."

# ---- Test case A ----
text_names_a = "Sarah went to the store with her brother Michael to buy some groceries."

prompt_names_a_zero = f"""{name_instruction}

Text: {text_names_a}
Names:"""

prompt_names_a_one = f"""{name_instruction}

Text: John met his old friend David at the coffee shop yesterday.
Names: John, David

Text: {text_names_a}
Names:"""

prompt_names_a_two = f"""{name_instruction}

Text: John met his old friend David at the coffee shop yesterday.
Names: John, David

Text: Emily and her husband Robert moved to a new city last year.
Names: Emily, Robert

Text: {text_names_a}
Names:"""

print("=== Names - Test case A ===")
print("ZERO-SHOT:", generate(prompt_names_a_zero))
print("\nONE-SHOT:", generate(prompt_names_a_one))
print("\nTWO-SHOT:", generate(prompt_names_a_two))

[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Names - Test case A ===


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ZERO-SHOT: List the person names occurring in the following texts.

Text: Sarah went to the store with her brother Michael to buy some groceries.
Names: Michael, Sarah, Michael's brother, Michael's girlfriend, Michael's mother


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ONE-SHOT: List the person names occurring in the following texts.

Text: John met his old friend David at the coffee shop yesterday.
Names: John, David

Text: Sarah went to the store with her brother Michael to buy some groceries.
Names: Sarah, Michael

Text: David went to the store with his brother

TWO-SHOT: List the person names occurring in the following texts.

Text: John met his old friend David at the coffee shop yesterday.
Names: John, David

Text: Emily and her husband Robert moved to a new city last year.
Names: Emily, Robert

Text: Sarah went to the store with her brother Michael to buy some groceries.
Names: Sarah, Michael

Text: David and his wife Sarah moved to a


In [15]:
# ---- Test case B ----
text_names_b = "The meeting was organized by Laura, and both Thomas and his assistant Rachel attended."

prompt_names_b_zero = f"""{name_instruction}

Text: {text_names_b}
Names:"""

prompt_names_b_one = f"""{name_instruction}

Text: John met his old friend David at the coffee shop yesterday.
Names: John, David

Text: {text_names_b}
Names:"""

prompt_names_b_two = f"""{name_instruction}

Text: John met his old friend David at the coffee shop yesterday.
Names: John, David

Text: Emily and her husband Robert moved to a new city last year.
Names: Emily, Robert

Text: {text_names_b}
Names:"""

print("=== Names - Test case B ===")
print("ZERO-SHOT:", generate(prompt_names_b_zero))
print("\nONE-SHOT:", generate(prompt_names_b_one))
print("\nTWO-SHOT:", generate(prompt_names_b_two))

[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Names - Test case B ===


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ZERO-SHOT: List the person names occurring in the following texts.

Text: The meeting was organized by Laura, and both Thomas and his assistant Rachel attended.
Names: Laura, Thomas, Rachel, and their assistant.

Text: The


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ONE-SHOT: List the person names occurring in the following texts.

Text: John met his old friend David at the coffee shop yesterday.
Names: John, David

Text: The meeting was organized by Laura, and both Thomas and his assistant Rachel attended.
Names: Thomas, Rachel

Text: The meeting was organized by Laura, and

TWO-SHOT: List the person names occurring in the following texts.

Text: John met his old friend David at the coffee shop yesterday.
Names: John, David

Text: Emily and her husband Robert moved to a new city last year.
Names: Emily, Robert

Text: The meeting was organized by Laura, and both Thomas and his assistant Rachel attended.
Names: Thomas, Rachel

Text: The meeting was organized by Laura, and


In [16]:
math_instruction = "This is a first grade math exam."

# ---- Test case A ----
prompt_math_a_zero = f"""{math_instruction}

Question: 11 + 22 =
Answer:"""

prompt_math_a_one = f"""{math_instruction}

Question: 5 + 3 =
Answer: 8

Question: 11 + 22 =
Answer:"""

prompt_math_a_two = f"""{math_instruction}

Question: 5 + 3 =
Answer: 8

Question: 14 + 10 =
Answer: 24

Question: 11 + 22 =
Answer:"""

print("=== Math - Test case A (expected: 33) ===")
print("ZERO-SHOT:", generate(prompt_math_a_zero, max_new_tokens=5))
print("\nONE-SHOT:", generate(prompt_math_a_one, max_new_tokens=5))
print("\nTWO-SHOT:", generate(prompt_math_a_two, max_new_tokens=5))

[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Math - Test case A (expected: 33) ===


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ZERO-SHOT: This is a first grade math exam.

Question: 11 + 22 =
Answer: 11 + 22 =



[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ONE-SHOT: This is a first grade math exam.

Question: 5 + 3 =
Answer: 8

Question: 11 + 22 =
Answer: 23

Question:

TWO-SHOT: This is a first grade math exam.

Question: 5 + 3 =
Answer: 8

Question: 14 + 10 =
Answer: 24

Question: 11 + 22 =
Answer: 31

Question:


In [17]:
prompt_math_b_zero = f"""{math_instruction}

Question: 14 + 10 =
Answer:"""

prompt_math_b_one = f"""{math_instruction}

Question: 5 + 3 =
Answer: 8

Question: 14 + 10 =
Answer:"""

prompt_math_b_two = f"""{math_instruction}

Question: 5 + 3 =
Answer: 8

Question: 11 + 22 =
Answer: 33

Question: 14 + 10 =
Answer:"""

print("=== Math - Test case B (expected: 24) ===")
print("ZERO-SHOT:", generate(prompt_math_b_zero, max_new_tokens=5))
print("\nONE-SHOT:", generate(prompt_math_b_one, max_new_tokens=5))
print("\nTWO-SHOT:", generate(prompt_math_b_two, max_new_tokens=5))

[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Math - Test case B (expected: 24) ===


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ZERO-SHOT: This is a first grade math exam.

Question: 14 + 10 =
Answer: 14

Question:


[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ONE-SHOT: This is a first grade math exam.

Question: 5 + 3 =
Answer: 8

Question: 14 + 10 =
Answer: 16

Question:

TWO-SHOT: This is a first grade math exam.

Question: 5 + 3 =
Answer: 8

Question: 11 + 22 =
Answer: 33

Question: 14 + 10 =
Answer: 14

Question:


## Results and analysis

I tested `gpt2-large` on three tasks (binary sentiment classification, person name
recognition, and two-digit addition), with two test cases per task, comparing
zero-shot, one-shot, and two-shot prompting.

### Task 1 -- Binary sentiment classification

| Test case | Zero-shot | One-shot | Two-shot |
|---|---|---|---|
| A (positive) | Fail -- model just repeated the input text, no label given | Correct -- "Positive" | Correct -- "Positive" |
| B (negative) | Fail -- no clear label given ("I'm not sure what to say") | Correct -- "Negative" | Fail -- "Positive" (incorrect) |

Zero-shot consistently failed because the model had no cue about the expected
answer format. A single example was already enough for the model to pick up the
pattern and classify correctly in both cases. Adding a second example did not
improve results -- in test case B it actually got worse, likely because the model
anchored to the label of the most recent example in the prompt ("Positive") rather
than reasoning about the new text's content (a form of recency bias).

### Task 2 -- Person name recognition

| Test case | Zero-shot | One-shot | Two-shot |
|---|---|---|---|
| A ("Sarah... brother Michael...") | Partial -- found both correct names (Sarah, Michael) but added hallucinated extras ("Michael's girlfriend", "Michael's mother") | Correct and clean -- "Sarah, Michael" | Correct and clean -- "Sarah, Michael" |
| B ("organized by Laura... Thomas... Rachel...") | Correct -- found all three names (Laura, Thomas, Rachel) | Incomplete -- only "Thomas, Rachel", missed Laura | Incomplete -- only "Thomas, Rachel", missed Laura |

Interestingly, this task shows the opposite pattern from sentiment: zero-shot
already performs reasonably well, since recognizing capitalized proper nouns is
a fairly "shallow" pattern the model has seen extensively during pretraining.
Few-shot examples mostly helped clean up the output format (no hallucinated
extras), but in test case B, few-shot actually caused the model to miss a name
that appeared in a different sentence position ("organized by Laura") than the
examples it was shown -- suggesting the model may be pattern-matching on sentence
structure rather than genuinely identifying all proper nouns.

### Task 3 -- Two-digit addition

| Test case | Zero-shot | One-shot | Two-shot |
|---|---|---|---|
| A (11 + 22 = 33) | Fail -- just repeated the question, no number given | Fail -- answered 23 (incorrect) | Fail -- answered 31 (incorrect) |
| B (14 + 10 = 24) | Fail -- answered 14 (just repeated an operand) | Fail -- answered 16 (incorrect) | Fail -- answered 14 (incorrect), even with a same-magnitude correct example (11+22=33) shown |

The model failed on every single attempt, regardless of shot count. Even giving
a correct example of similar magnitude in the two-shot case did not help. This
confirms that `gpt2-large` has no real arithmetic capability -- it predicts the
statistically most plausible next token based on patterns seen during training,
rather than actually computing a result. Few-shot examples can teach the model
the expected answer format (a number after "Answer:"), but not the underlying
computation itself.

### Overall conclusion

Performance depends heavily on the type of task, not just the number of examples
given:
- For tasks that are essentially surface-level language pattern recognition
  (person name recognition), the model already performs reasonably in zero-shot,
  and few-shot mainly helps constrain the output format.
- For abstract classification labels (sentiment), at least one example is needed
  for the model to understand what output is expected, but more examples do not
  guarantee improvement -- they can introduce biases toward the most recent
  example shown.
- For tasks requiring actual computation (arithmetic), no number of examples
  helped -- the model lacks the underlying capability entirely, since generative
  language models predict plausible-looking tokens rather than executing real
  calculations.

**Submit this exercise by submitting your code and your answers to the above questions as comments on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**